In [0]:
# Create widgets for landing_timestamp and table_name
dbutils.widgets.text("landing_timestamp", "", "landing timestamp")
dbutils.widgets.text("table_name", "", "table Name")
dbutils.widgets.text("merge_keys", "", "merge keys")
dbutils.widgets.text("load_type", "", "load type")

# Get widget values into variables
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"azuredatabricks_medical.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql import functions as F
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp())\
            .withColumn("update_timestamp", F.current_timestamp())

In [0]:
target_table = f"azuredatabricks_medical.silver.{table_name}"
source_record_count = updated_df.count()

print("="*80)
print(f"LOAD CONFIGURATION")
print("="*80)
print(f"  Load Type       : {load_type}")
print(f"  Target Table    : {target_table}")
print(f"  Source Records  : {source_record_count}")
print(f"  Source Columns  : {updated_df.columns}")
print(f"  Merge Keys      : {merge_keys}")
print("="*80)

if load_type == "FULL":
    print(f"\n[FULL LOAD] Overwriting target table '{target_table}' with {source_record_count} records...")
    print(f"[FULL LOAD] Equivalent SQL:")
    print(f"  INSERT OVERWRITE TABLE {target_table}")
    print(f"  SELECT * FROM source_dataframe")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[FULL LOAD] SUCCESS — {source_record_count} records written to '{target_table}'.")

elif load_type == "APPEND":
    print(f"\n[APPEND] Appending {source_record_count} records to '{target_table}'...")
    print(f"[APPEND] Equivalent SQL:")
    print(f"  INSERT INTO TABLE {target_table}")
    print(f"  SELECT * FROM source_dataframe")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[APPEND] SUCCESS — {source_record_count} records appended to '{target_table}'.")

elif load_type == "MERGE":
    from delta.tables import DeltaTable

    merge_key_list = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_key_list])

    print(f"\n[MERGE] Merge keys       : {merge_key_list}")
    print(f"[MERGE] Merge condition  : {merge_condition}")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"[MERGE] Target table exists: {table_exists}")

    if table_exists:
        delta_table = DeltaTable.forName(spark, target_table)
        target_record_count = spark.read.table(target_table).count()
        print(f"[MERGE] Existing target record count: {target_record_count}")

        # Update all columns except merge keys and insert_timestamp (preserve original insert time)
        update_set = {c: f"source.{c}" for c in updated_df.columns if c not in merge_key_list and c != "insert_timestamp"}
        update_columns = list(update_set.keys())

        # Insert all columns for new records
        insert_values = {c: f"source.{c}" for c in updated_df.columns}

        print(f"\n[MERGE] Equivalent SQL:")
        print(f"  MERGE INTO {target_table} AS target")
        print(f"  USING source_dataframe AS source")
        print(f"  ON {merge_condition}")
        print(f"  WHEN MATCHED THEN UPDATE SET")
        for col_name in update_columns:
            print(f"    {col_name} = source.{col_name}")
        print(f"  WHEN NOT MATCHED THEN INSERT ({', '.join(insert_values.keys())})")
        print(f"    VALUES ({', '.join(insert_values.values())})")

        print(f"\n[MERGE] Executing merge...")
        delta_table.alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(set=update_set) \
         .whenNotMatchedInsert(values=insert_values) \
         .execute()

        final_count = spark.read.table(target_table).count()
        new_records = final_count - target_record_count
        updated_records = source_record_count - new_records
        print(f"\n[MERGE] SUCCESS — Merge completed.")
        print(f"  Records before merge : {target_record_count}")
        print(f"  Records after merge  : {final_count}")
        print(f"  New records inserted : {new_records}")
        print(f"  Records updated      : {updated_records}")
    else:
        print(f"[MERGE] Target table does not exist. Performing initial full load...")
        print(f"[MERGE] Equivalent SQL:")
        print(f"  CREATE TABLE {target_table} AS")
        print(f"  SELECT * FROM source_dataframe")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[MERGE] SUCCESS — Initial load of {source_record_count} records into '{target_table}'.")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print("\n" + "="*80)
print("LOAD COMPLETE")
print("="*80)

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))